[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HesusG/dammg-supervisado/blob/main/notebooks/salas/S2_hoja_repaso.ipynb)

# Hoja de trabajo: indaga en tu estación

**Al terminar podrás** describir con una gráfica cómo se usa una estación de MiBici, leer sus columnas
y comparar una recta hecha por ti contra la regla "hace una semana" con el MAE de julio.

---

**De:** Coordinación de operaciones, MiBici<br>
**Asunto:** ¿vale la pena cambiar la regla de la semana pasada?

Buen día. Hoy pronosticamos las salidas de cada estación con la regla "las mismas que hace una semana".
Antes de cambiar nada queremos conocer bien **una estación**: a qué horas se usa, si el fin de semana cambia
y si una recta la predice mejor que la regla. Escojan una estación, grafíquenla, ajusten la recta con octubre a junio,
mídanla en julio con el MAE y respondan en tres renglones.

---

## Las columnas de MiBici

Cada fila es una estación en una hora. El ejemplo es la estación 51 el lunes 27 de julio de 2026 a las 8:00.
La última columna dice si el dato ya se conoce a las 7:59, antes de la hora que queremos predecir.
Si dice **no**, usarla es **fuga**: el modelo estaría viendo la respuesta.

| Columna | Qué significa | Ejemplo | ¿Se conoce antes de esa hora? |
|---|---|---|---|
| `station_id` | número de la estación | 51 | sí |
| `hour` | fecha y hora de la fila | 2026-07-27 08:00 | sí |
| `hour_of_day` | hora del día, de 0 a 23 | 8 | sí |
| `day_of_week` | día de la semana: 0 es lunes, 6 es domingo | 0 | sí |
| `is_weekend` | 1 si es sábado o domingo, 0 si no | 0 | sí |
| `month` | mes, de 1 a 12 | 7 | sí |
| `capacity` | cuántos anclajes tiene la estación | 39 | sí |
| `departures` | bicis que salieron en esa hora: **lo que queremos predecir** | 34 | no: es la respuesta |
| `arrivals` | bicis que llegaron en esa hora | 9 | no: pasa durante la hora |
| `net_flow` | salidas menos llegadas en esa hora | 25 | no: pasa durante la hora |
| `future_net` | salidas menos llegadas, sumadas en esa hora y las 2 siguientes | 46 | no: es el futuro |
| `departures_lag_24h` | salidas a la misma hora, hace 24 horas | 12 | sí |
| `departures_lag_168h` | salidas a la misma hora, hace una semana (168 horas) | 32 | sí |
| `departures_mean_24h` | promedio de salidas por hora en el último día | 9.3 | sí |
| `departures_mean_168h` | promedio de salidas por hora en la última semana | 13.0 | sí |
| `net_lag_24h` | salidas menos llegadas a la misma hora, hace 24 horas | 0 | sí |
| `net_lag_168h` | salidas menos llegadas a la misma hora, hace una semana | 21 | sí |
| `net_mean_24h` | promedio de salidas menos llegadas en el último día | -1.8 | sí |
| `particion` | a qué parte pertenece la fila: train (octubre a junio), val (julio) o test (agosto y septiembre) | val | no es un dato para predecir |

Corran las celdas de abajo hasta que aparezca `✅ entorno listo`.

In [ ]:
# @title Preparación del entorno: solo córrela, no hace falta leerla
# Descarga los datos y dammg_utils.py, revisa que no lleguen dañados y avisa con ✅ entorno listo.
import hashlib, importlib.util, os, pathlib, subprocess, sys, urllib.request

BASE_URL = "https://hesusg.github.io/dammg-supervisado/datos/"
# Versiones fijas: copia generada de notebooks/requirements.txt (fuente única)
REQUISITOS = {
    'numpy': 'numpy==2.5.4',
    'pandas': 'pandas==3.0.6',
    'sklearn': 'scikit-learn==1.9.1',
    'matplotlib': 'matplotlib==3.11.2',
    'ipywidgets': 'ipywidgets==8.1.9',
    'pyarrow': 'pyarrow==26.0.0',
}
SHA256 = {
    'mibici_curso.parquet': 'a5fd844c57bbf220f0ddfc27dc965ef8fb8438bd6c1167709bde7ccc1dafa6d8',
    'dammg_utils.py': '7499b8695cb94d4dd72133e197d95a143e8b05b82f1d91928f6a8a30fa0f62d0',
}
EN_EVALUADOR = os.environ.get("DAMMG_GRADER") == "1"
# Umbral del evento |future_net| (percentil 85 de train), congelado para todo el curso
UMBRAL_EVENTO = 7.0


def _sha256(ruta):
    return hashlib.sha256(pathlib.Path(ruta).read_bytes()).hexdigest()


def _copia_local(nombre):
    """Busca una copia local: DAMMG_DATA_DIR o el repo del curso (data/curso, notebooks)."""
    carpetas = []
    if os.environ.get("DAMMG_DATA_DIR"):
        carpetas.append(pathlib.Path(os.environ["DAMMG_DATA_DIR"]))
    aqui = pathlib.Path.cwd().resolve()
    for p in [aqui, *aqui.parents][:5]:
        carpetas += [p / "data" / "curso", p / "notebooks", p]
    for c in carpetas:
        if (c / nombre).is_file():
            return c / nombre
    return None


def _obtener(nombre, esperado):
    """Copia local si existe; si no, descarga del sitio del curso y verifica el sha256."""
    if EN_EVALUADOR:
        # Dentro del evaluador: nada de descargar; los datos ya están en /data
        return pathlib.Path("/data") / nombre
    local = _copia_local(nombre)
    if local is not None:
        if _sha256(local) != esperado:
            print(f"⚠ {nombre}: la copia local no coincide con el sha256 publicado (¿versión en desarrollo?)")
        return local
    destino = pathlib.Path("datos_dammg") / nombre
    destino.parent.mkdir(exist_ok=True)
    if not destino.exists() or _sha256(destino) != esperado:
        print(f"Descargando {nombre} ...")
        try:
            urllib.request.urlretrieve(BASE_URL + nombre, destino)
        except OSError as e:   # HTTPError, sin internet, etc.
            raise RuntimeError(
                f"No pude descargar {nombre} ({e}). Revisa tu conexión y vuelve a ejecutar esta celda. "
                "Si sigue fallando, avisa en el foro de Moodle: puede que el archivo aún no esté publicado."
            ) from None
    assert _sha256(destino) == esperado, f"{nombre} llegó incompleto o alterado: bórralo y vuelve a ejecutar."
    return destino


if not EN_EVALUADOR:
    faltan = [pin for modulo, pin in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
    if faltan:
        print("Instalando:", ", ".join(faltan))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
RUTAS = {nombre: _obtener(nombre, esperado) for nombre, esperado in SHA256.items()}

sys.path.insert(0, str(RUTAS["dammg_utils.py"].parent))
RUTA_DATOS = RUTAS["mibici_curso.parquet"]

import dammg_utils as du
du.entorno_listo()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

du.estilo_dammg()
df = pd.read_parquet(RUTA_DATOS)
train = df[df.particion == "train"]   # octubre a junio: para ajustar
julio = df[df.particion == "val"]     # julio: para medir

## Paso 1. Escojan su estación

Cambien el número. Hay 40 estaciones; si no saben cuál, prueben con 51, 271, 11 o 194.

In [ ]:
estacion = 51                                         # cambien el número por el de su estación
train_mia = train[train.station_id == estacion]       # octubre a junio, su estación
julio_mia = julio[julio.station_id == estacion]       # julio, su estación
print(f"Estación {estacion}: {len(train_mia):,} horas en train, {len(julio_mia):,} en julio")
print(f"Salidas por hora en train: promedio {train_mia.departures.mean():.1f}, máximo {train_mia.departures.max()}")

## Paso 2. ¿A qué horas se usa?

Promedio de salidas por hora del día, entre semana y en fin de semana. Antes de correr, piensen: ¿esperan un pico en la mañana, en la tarde o en los dos?

In [ ]:
perfil = train_mia.groupby(["hour_of_day", "is_weekend"]).departures.mean().reset_index()
perfil["día"] = perfil.is_weekend.map({0: "entre semana", 1: "fin de semana"})
sns.lineplot(data=perfil, x="hour_of_day", y="departures", hue="día", marker="o")
plt.title(f"Estación {estacion}: salidas promedio por hora (octubre a junio)")
plt.xlabel("hora del día"); plt.ylabel("salidas por hora")
plt.show()

Anoten: ¿a qué hora es el pico entre semana? ¿Cuántas bicis salen en ese pico? ¿El fin de semana se parece?

## Paso 3. ¿Hace una semana sirve para predecir hoy?

Cada punto es el promedio de las salidas de hoy para cada valor de hace una semana, en su estación.
Si los puntos forman una línea, una recta tiene sentido.

In [ ]:
puntos = train_mia.groupby("departures_lag_168h").departures.mean().reset_index()
sns.scatterplot(data=puntos, x="departures_lag_168h", y="departures")
plt.title(f"Estación {estacion}: hoy contra hace una semana (octubre a junio)")
plt.xlabel("salidas hace una semana, misma hora"); plt.ylabel("salidas hoy (promedio)")
plt.show()

## Paso 4. Su recta

Escojan **una** columna de la tabla que diga **sí** en la última columna. Luego completen las dos líneas: quiten el `#` y cambien cada `___`.

In [ ]:
columna = "departures_lag_168h"     # pueden escoger otra columna de la tabla

# recta = LinearRegression().fit(___, train_mia.departures)   # pista: la columna va con doble corchete: train_mia[[columna]]
# pred = recta.predict(___)                                   # pista: la misma columna, pero de julio_mia

## Paso 5. La medición

Calculen el MAE en julio de su recta y de la regla "hace una semana". Más chico es mejor.

In [ ]:
# mae_recta = mean_absolute_error(julio_mia.departures, ___)   # pista: lo que predijo su recta
# mae_regla = mean_absolute_error(julio_mia.departures, ___)   # pista: la regla es julio_mia.departures_lag_168h
# print(f"MAE en julio: recta {mae_recta:.2f}   regla {mae_regla:.2f}")

Un día de julio en su estación: lo real contra su recta y la regla.
Esta celda usa `pred` del paso 4: si sale `NameError`, completen primero las dos líneas del paso 4 y córranlo.

In [ ]:
dia = julio_mia.assign(recta=pred)[julio_mia.hour.dt.day == 27]   # lunes 27 de julio
dia = dia.rename(columns={"departures": "real", "departures_lag_168h": "regla"})
largo = dia.melt(id_vars="hour_of_day", value_vars=["real", "recta", "regla"], var_name="serie", value_name="bicis")
sns.lineplot(data=largo, x="hour_of_day", y="bicis", hue="serie", marker="o")
plt.title(f"Estación {estacion}, lunes 27 de julio")
plt.show()

## Paso 6. La respuesta a operaciones

Escriban en la celda de abajo (doble clic), con sus números:

1. **Cómo es la estación:** el pico es a las ___ con ___ bicis; el fin de semana ___.
2. **El número:** en julio, la recta con `___` falla en promedio por ___ bicis por hora; la regla, por ___.
3. **Recomendación y riesgo:** ___ (¿cambiamos la regla en esta estación? ¿qué columna no deberíamos usar y por qué?).

*Su respuesta aquí.*

## Preguntas (escriban la letra debajo de cada una)

**1. ¿Qué dice la columna `departures_lag_24h`?**

- a) Las salidas de las próximas 24 horas
- b) Las salidas de la misma estación a la misma hora, ayer
- c) Cuántas horas tardó la bici en volver
- d) El promedio de salidas del día

**Su respuesta (doble clic para escribir):** ___

**2. Queremos predecir las salidas de las 8:00. ¿Cuál columna es fuga?**

- a) `departures_lag_168h`
- b) `hour_of_day`
- c) `future_net`
- d) `capacity`

**Su respuesta (doble clic para escribir):** ___

**3. En su gráfica del paso 2, ¿qué quiere decir que el pico entre semana sea en la mañana?**

- a) Que la gente sale de esa estación para ir a trabajar o estudiar
- b) Que la estación está cerrada en la tarde
- c) Que el modelo está mal
- d) Nada: los picos son al azar

**Su respuesta (doble clic para escribir):** ___

**4. Una recta tiene MAE 4.2 en julio y la regla 4.1. ¿Qué le dicen a operaciones?**

- a) Cambiemos ya a la recta
- b) Casi empatan: con una sola columna no vale la pena cambiar la regla
- c) El MAE no sirve para comparar
- d) Hay que medir en agosto antes de decidir nada

**Su respuesta (doble clic para escribir):** ___